# Milestone 2 - Data Cleaning and Preparation
**Author:** Stephen Omengo Owuor

Re-runnable pipeline: raw CSVs in `data/raw/` -> clean tables in `data/clean/`. Raw files are never modified.

**Steps:** 1) load and validate, 2) profile issues, 3) fix each issue with before/after counts, 4) build features, 5) validate and save.

In [1]:
import pandas as pd, numpy as np
from pathlib import Path

RAW, CLEAN = Path("../data/raw"), Path("../data/clean")
CLEAN.mkdir(parents=True, exist_ok=True)

sales = pd.read_csv(RAW / "sales_transactions.csv", parse_dates=["date"])
products = pd.read_csv(RAW / "products.csv")
stores = pd.read_csv(RAW / "stores.csv", parse_dates=["open_date"])
customers = pd.read_csv(RAW / "customers.csv", parse_dates=["signup_date"])
calendar = pd.read_csv(RAW / "calendar.csv", parse_dates=["date"])
promotions = pd.read_csv(RAW / "promotions.csv", parse_dates=["start_date", "end_date"])

log = []   # one row per cleaning step, saved at the end
def record(step, issue, before, after, action):
    log.append(dict(step=step, issue=issue, rows_affected=before - after if isinstance(before, int) else before,
                    action=action))
print({n: len(d) for n, d in dict(sales=sales, products=products, stores=stores, customers=customers,
                                  calendar=calendar, promotions=promotions).items()})

{'sales': 302680, 'products': 98, 'stores': 10, 'customers': 5000, 'calendar': 731, 'promotions': 60}


## 1. Validate structure

In [2]:
print(sales.dtypes, "\n")
print("Date range:", sales.date.min().date(), "to", sales.date.max().date())

# Foreign-key checks: every code in sales must exist in its reference table
print("Unknown store_id:", (~sales.store_id.isin(stores.store_id)).sum())
print("Unknown product_id:", (~sales.product_id.isin(products.product_id)).sum())
print("Unknown customer_id:", (~sales.customer_id.dropna().isin(customers.customer_id)).sum())

line_id                    int64
date              datetime64[ns]
store_id                  object
transaction_id            object
payment_method            object
customer_id               object
product_id                object
quantity                   int64
unit_price                 int64
promo_id                  object
discount_pct               int64
line_total               float64
dtype: object 

Date range: 2023-07-01 to 2025-06-30
Unknown store_id: 0
Unknown product_id: 0
Unknown customer_id: 0


## 2. Profile the issues

In [3]:
# Line total should equal quantity x catalogue price x (1 - discount). Use it to judge which column is wrong.
cat_price = products.set_index("product_id").unit_price
sales["_cat_price"] = sales.product_id.map(cat_price)
sales["_expected"] = (sales.quantity.abs() * sales._cat_price * (1 - sales.discount_pct / 100)).round(2)

profile = pd.Series({
    "rows": len(sales),
    "missing payment_method": sales.payment_method.isna().sum(),
    "missing customer_id (guest checkout)": sales.customer_id.isna().sum(),
    "missing promo_id (no promotion)": sales.promo_id.isna().sum(),
    "exact duplicate lines (ignoring line_id)": sales.duplicated(subset=[c for c in sales.columns if c not in ("line_id",) and not c.startswith("_")]).sum(),
    "negative quantity": (sales.quantity < 0).sum(),
    "unit_price differs from catalogue": (sales.unit_price != sales._cat_price).sum(),
    "line_total does not match expected": ((sales.line_total - sales._expected).abs() > 0.01).sum(),
})
profile

rows                                        302680
missing payment_method                        1212
missing customer_id (guest checkout)        126744
missing promo_id (no promotion)             284468
exact duplicate lines (ignoring line_id)       604
negative quantity                              302
unit_price differs from catalogue              151
line_total does not match expected               0
dtype: int64

In [4]:
# Is payment_method one value per transaction? (decides whether we can fill gaps from sibling lines)
per_txn = sales.dropna(subset=["payment_method"]).groupby("transaction_id").payment_method.nunique()
print("Transactions with more than one payment method:", (per_txn > 1).sum())

# What do the price outliers look like?
bad = sales[sales.unit_price != sales._cat_price]
print("Price / catalogue price ratios:", (bad.unit_price / bad._cat_price).round(2).value_counts().to_dict())

Transactions with more than one payment method: 0
Price / catalogue price ratios: {10.0: 151}


**Decisions**
- Missing `customer_id` and `promo_id` are **not errors**: they mean a guest checkout and no promotion. Keep them and add flags.
- Payment method is one value per transaction, so missing values can be filled from the other lines of the same transaction.
- Price outliers are exactly 10x the catalogue price while `line_total` is still correct, so the price was mis-keyed: restore it from the catalogue.
- Negative quantities: `line_total` is still positive, so the sign was entered wrongly (not a return): take the absolute value.
- Exact duplicate lines: drop them.

## 3. Fix each issue

In [5]:
# Step 1 - duplicates
cols = [c for c in sales.columns if c != "line_id" and not c.startswith("_")]
before = len(sales)
sales = sales.drop_duplicates(subset=cols, keep="first").copy()
print(f"Duplicates removed: {before - len(sales):,}  ({before:,} -> {len(sales):,} rows)")
record("1", "Exact duplicate lines", before, len(sales), "Dropped, kept first occurrence")

Duplicates removed: 604  (302,680 -> 302,076 rows)


In [6]:
# Step 2 - negative quantities
n = int((sales.quantity < 0).sum())
sales["quantity"] = sales.quantity.abs()
print(f"Negative quantities fixed: {n}  -> remaining: {(sales.quantity < 0).sum()}")
record("2", "Negative quantity", n, 0, "Took absolute value (line_total was positive)")

Negative quantities fixed: 302  -> remaining: 0


In [7]:
# Step 3 - unit price outliers
n = int((sales.unit_price != sales._cat_price).sum())
sales["unit_price"] = sales._cat_price
print(f"Prices restored from catalogue: {n}  -> remaining: {(sales.unit_price != sales._cat_price).sum()}")
record("3", "Unit price 10x catalogue", n, 0, "Replaced with catalogue price")

Prices restored from catalogue: 151  -> remaining: 0


In [8]:
# Step 4 - missing payment method
n = int(sales.payment_method.isna().sum())
sibling = sales.dropna(subset=["payment_method"]).drop_duplicates("transaction_id").set_index("transaction_id").payment_method
sales["payment_method"] = sales.payment_method.fillna(sales.transaction_id.map(sibling))
still = int(sales.payment_method.isna().sum())
sales["payment_method"] = sales.payment_method.fillna("Unknown")
print(f"Missing payment methods: {n}; filled from same transaction: {n - still}; labelled 'Unknown': {still}")
record("4", "Missing payment_method", n, still, "Filled from other lines in same transaction; rest set to 'Unknown'")

Missing payment methods: 1208; filled from same transaction: 1121; labelled 'Unknown': 87


## 4. Build features

In [9]:
sales["has_customer_id"] = sales.customer_id.notna()
sales["has_promo"] = sales.promo_id.notna()

clean = (sales.drop(columns=["_cat_price", "_expected"])
    .merge(products[["product_id", "product_name", "category", "unit_cost"]], on="product_id", how="left")
    .merge(stores[["store_id", "store_name", "county", "location_type"]], on="store_id", how="left")
    .merge(customers[["customer_id", "age_group", "loyalty_tier"]], on="customer_id", how="left")
    .merge(calendar[["date", "day_of_week", "is_weekend", "is_holiday", "holiday_name",
                     "is_payday_window", "is_school_opening"]], on="date", how="left"))

clean["revenue"] = clean.line_total
clean["cost"] = (clean.quantity * clean.unit_cost).round(2)
clean["gross_profit"] = (clean.revenue - clean.cost).round(2)
clean["year_month"] = clean.date.dt.to_period("M").astype(str)
clean.shape

(302076, 32)

## 5. Validate and save

In [10]:
expected = (clean.quantity * clean.unit_price * (1 - clean.discount_pct / 100)).round(2)
checks = {
    "no duplicate lines": not clean.duplicated(subset=[c for c in clean.columns if c != "line_id"]).any(),
    "no missing payment_method": clean.payment_method.notna().all(),
    "all quantities positive": (clean.quantity > 0).all(),
    "line_total reconciles with qty x price x discount": ((clean.line_total - expected).abs() <= 0.01).all(),
    "no missing store / product / calendar joins": clean[["store_name", "category", "day_of_week"]].notna().all().all(),
    "dates inside 2023-07-01 to 2025-06-30": clean.date.between("2023-07-01", "2025-06-30").all(),
}
for k, v in checks.items():
    print("PASS" if v else "FAIL", "-", k)
assert all(checks.values())

PASS - no duplicate lines
PASS - no missing payment_method
PASS - all quantities positive
PASS - line_total reconciles with qty x price x discount
PASS - no missing store / product / calendar joins
PASS - dates inside 2023-07-01 to 2025-06-30


In [11]:
clean.to_csv(CLEAN / "sales_clean.csv", index=False)
pd.DataFrame(log).to_csv(CLEAN / "cleaning_log.csv", index=False)
print("Saved", len(clean), "rows to data/clean/sales_clean.csv")
pd.DataFrame(log)

Saved 302076 rows to data/clean/sales_clean.csv


,step,issue,rows_affected,action
0,1,Exact duplicate lines,604,"Dropped, kept first occurrence"
1,2,Negative quantity,302,Took absolute value (line_total was positive)
2,3,Unit price 10x catalogue,151,Replaced with catalogue price
3,4,Missing payment_method,1121,Filled from other lines in same transaction; r...


## Summary
Fill in your own words after running: what each issue was, how many rows it touched, and why you chose that fix.
Raw rows: 302,680. Clean rows: see the output above.